# Retail Data Integration — Data Cleaning

## Objective

This notebook converts the inspected raw FMCG retail dataset into a clean, analysis-ready dataset for SQL and Excel.

The original file remains unchanged. All transformations are applied to a separate DataFrame and exported to the `data/cleaned` folder.

## 1. Load the Raw Dataset

The raw CSV is loaded into pandas, and a separate copy is created for cleaning.

In [1]:
import pandas as pd
import numpy as np

raw_file_path = r"C:\Users\janakiram\Documents\Data_Analytics_Portfolio\PROJECTS\Retail_Data_Integration\data\raw\Indian FMCG Retail Sales  Customer  Inventory (2024).csv"

df = pd.read_csv(raw_file_path)
clean_df = df.copy()

print("Raw shape:", df.shape)
print("Cleaning copy shape:", clean_df.shape)

Raw shape: (100000, 21)
Cleaning copy shape: (100000, 21)


## 2. Create a Unique Transaction Key

`Invoice_ID` contains 62 identifier collisions. Both records in each collision contain valid-looking but different transactions.

A surrogate `Transaction_ID` is created to provide a reliable unique key while preserving the original invoice reference.

In [3]:
clean_df.insert(
    0,
    "Transaction_ID",
    [f"TXN_{number:06d}" for number in range(1, len(clean_df) + 1)]
)

print("Transaction IDs unique:", clean_df["Transaction_ID"].is_unique)
print(clean_df[["Transaction_ID", "Invoice_ID"]].head())

Transaction IDs unique: True
  Transaction_ID  Invoice_ID
0     TXN_000001    58018430
1     TXN_000002    48157952
2     TXN_000003    23283831
3     TXN_000004    53537460
4     TXN_000005    55348596


## 3. Standardize Column Names and Data Types

The percentage column is renamed for easier use in SQL. Identifier, date, age, and binary fields are converted to appropriate data types.

In [5]:
clean_df = clean_df.rename(
    columns={"Margin_%": "Margin_Pct"}
)

clean_df["Transaction_ID"] = clean_df["Transaction_ID"].astype("string")
clean_df["Invoice_ID"] = clean_df["Invoice_ID"].astype("string")
clean_df["Invoice_Date"] = pd.to_datetime(clean_df["Invoice_Date"])

clean_df["Customer_Age"] = clean_df["Customer_Age"].astype("Int64")
clean_df["Loyalty_Flag"] = clean_df["Loyalty_Flag"].astype("Int8")

clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 22 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   Transaction_ID   100000 non-null  string        
 1   Invoice_ID       100000 non-null  string        
 2   Invoice_Date     100000 non-null  datetime64[ns]
 3   City             100000 non-null  object        
 4   Store_Format     100000 non-null  object        
 5   Category         100000 non-null  object        
 6   Brand            100000 non-null  object        
 7   Channel          100000 non-null  object        
 8   Payment_Mode     100000 non-null  object        
 9   Units            100000 non-null  int64         
 10  Cost_Price       100000 non-null  float64       
 11  Selling_Price    100000 non-null  float64       
 12  Revenue          100000 non-null  float64       
 13  Cost             100000 non-null  float64       
 14  Margin           1000

## 4. Handle Missing Customer Attributes

`Customer_Age` has a high missing rate, so missing ages are retained rather than replaced with an estimated value.

A separate `Age_Group` field is created for grouped analysis, with missing ages labelled as `Unknown`.

Gender codes are expanded into readable labels, and missing gender values are classified as `Unknown`. The binary loyalty flag is retained, while a readable loyalty-status column is added.m

In [7]:
age_bins = [17, 24, 34, 44, 54, 64]
age_labels = [
    "18-24",
    "25-34",
    "35-44",
    "45-54",
    "55-64"
]

clean_df["Age_Group"] = pd.cut(
    clean_df["Customer_Age"],
    bins=age_bins,
    labels=age_labels
)

clean_df["Age_Group"] = (
    clean_df["Age_Group"]
    .astype("string")
    .fillna("Unknown")
)

gender_mapping = {
    "M": "Male",
    "F": "Female",
    "O": "Other"
}

clean_df["Customer_Gender"] = (
    clean_df["Customer_Gender"]
    .map(gender_mapping)
    .fillna("Unknown")
    .astype("string")
)

loyalty_mapping = {
    1: "Member",
    0: "Non-Member"
}

clean_df["Loyalty_Status"] = (
    clean_df["Loyalty_Flag"]
    .map(loyalty_mapping)
    .astype("string")
)

In [8]:
print("Missing customer ages:", clean_df["Customer_Age"].isna().sum())
print("Missing customer genders:", clean_df["Customer_Gender"].isna().sum())

print("\nAge-group distribution:")
print(clean_df["Age_Group"].value_counts(dropna=False))

print("\nGender distribution:")
print(clean_df["Customer_Gender"].value_counts(dropna=False))

print("\nLoyalty-status distribution:")
print(clean_df["Loyalty_Status"].value_counts(dropna=False))

Missing customer ages: 40081
Missing customer genders: 0

Age-group distribution:
Age_Group
Unknown    40081
55-64      12891
45-54      12770
35-44      12731
25-34      12665
18-24       8862
Name: count, dtype: Int64

Gender distribution:
Customer_Gender
Unknown    100000
Name: count, dtype: Int64

Loyalty-status distribution:
Loyalty_Status
Non-Member    70280
Member        29720
Name: count, dtype: Int64


In [9]:
print([
    repr(value)
    for value in df["Customer_Gender"].dropna().unique()
])

["'M'", "'F'", "'O'"]


In [10]:
clean_df["Customer_Gender"] = (
    df["Customer_Gender"]
    .astype("string")
    .str.strip()
    .str.upper()
    .map(gender_mapping)
    .fillna("Unknown")
    .astype("string")
)

In [11]:
print(clean_df["Customer_Gender"].value_counts(dropna=False))
print(
    "Missing customer genders:",
    clean_df["Customer_Gender"].isna().sum()
)

Customer_Gender
Male       44920
Female     44905
Other       5127
Unknown     5048
Name: count, dtype: Int64
Missing customer genders: 0


### Gender Standardization Note

The original gender codes contained hidden whitespace, which initially prevented direct mapping to readable labels.

The values were rebuilt from the untouched raw DataFrame, trimmed, converted to uppercase, and mapped to `Male`, `Female`, and `Other`. Missing values were labelled `Unknown`.

## 5. Standardize Categorical Text

Leading and trailing whitespace is removed from categorical text columns. This prevents values that look identical from being treated as separate categories in SQL, Python, or Excel.

In [13]:
text_cleanup_columns = [
    "City",
    "Store_Format",
    "Category",
    "Brand",
    "Channel",
    "Payment_Mode"
]

for column in text_cleanup_columns:
    clean_df[column] = (
        clean_df[column]
        .astype("string")
        .str.strip()
    )

In [14]:
remaining_outer_spaces = pd.Series({
    column: (
        clean_df[column].notna()
        & clean_df[column].ne(clean_df[column].str.strip())
    ).sum()
    for column in text_cleanup_columns
})

display(
    remaining_outer_spaces.to_frame(
        "Values_With_Outer_Spaces"
    )
)

print("\nDistinct values after trimming:")

for column in text_cleanup_columns:
    print(f"{column}: {clean_df[column].nunique()}")

,Values_With_Outer_Spaces
City,0
Store_Format,0
Category,0
Brand,0
Channel,0
Payment_Mode,0



Distinct values after trimming:
City: 8
Store_Format: 3
Category: 8
Brand: 8
Channel: 3
Payment_Mode: 4


## 6. Create Analysis-Ready Date Fields

Additional date attributes are derived from the invoice timestamp. These fields simplify grouping and filtering in SQL and Excel while retaining the original timestamp.

In [17]:
clean_df["Invoice_Date_Only"] = (
    clean_df["Invoice_Date"].dt.normalize()
)

clean_df["Invoice_Year"] = (
    clean_df["Invoice_Date"].dt.year.astype("int16")
)

clean_df["Invoice_Quarter"] = (
    clean_df["Invoice_Date"].dt.to_period("Q").astype("string")
)

clean_df["Invoice_Month"] = (
    clean_df["Invoice_Date"].dt.to_period("M").astype("string")
)

clean_df["Month_Number"] = (
    clean_df["Invoice_Date"].dt.month.astype("int8")
)

clean_df["Month_Name"] = (
    clean_df["Invoice_Date"].dt.month_name().astype("string")
)

clean_df["Transaction_Hour"] = (
    clean_df["Invoice_Date"].dt.hour.astype("int8")
)

## 7. Create Inventory-Replenishment Fields

Inventory records at or below their reorder level are flagged for replenishment. A stock-buffer field measures how far current inventory is above or below the reorder threshold.

In [18]:
clean_df["Stock_Buffer"] = (
    clean_df["Stock_On_Hand"]
    - clean_df["Reorder_Level"]
)

clean_df["Reorder_Flag"] = (
    clean_df["Stock_On_Hand"]
    <= clean_df["Reorder_Level"]
).astype("int8")

clean_df["Inventory_Status"] = np.where(
    clean_df["Reorder_Flag"].eq(1),
    "Reorder Required",
    "Sufficient Stock"
)

clean_df["Inventory_Status"] = (
    clean_df["Inventory_Status"].astype("string")
)

In [19]:
print("Cleaned shape:", clean_df.shape)

print("\nDate range:")
print(
    clean_df["Invoice_Date_Only"].min(),
    "to",
    clean_df["Invoice_Date_Only"].max()
)

print("\nInvoice months:", clean_df["Invoice_Month"].nunique())
print("Invoice quarters:", clean_df["Invoice_Quarter"].nunique())

print("\nInventory status:")
print(clean_df["Inventory_Status"].value_counts())

print("\nReorder flag:")
print(clean_df["Reorder_Flag"].value_counts())

print(
    "\nStatus and flag agree:",
    (
        clean_df["Inventory_Status"].eq("Reorder Required")
        == clean_df["Reorder_Flag"].eq(1)
    ).all()
)

Cleaned shape: (100000, 34)

Date range:
2024-01-01 00:00:00 to 2024-12-30 00:00:00

Invoice months: 12
Invoice quarters: 4

Inventory status:
Inventory_Status
Sufficient Stock    98271
Reorder Required     1729
Name: count, dtype: Int64

Reorder flag:
Reorder_Flag
0    98271
1     1729
Name: count, dtype: int64

Status and flag agree: True


## 8. Arrange the Cleaned Dataset

Columns are placed in a logical business order: identifiers and dates, transaction attributes, financial measures, inventory fields, and customer attributes.

In [20]:
column_order = [
    "Transaction_ID",
    "Invoice_ID",
    "Invoice_Date",
    "Invoice_Date_Only",
    "Invoice_Year",
    "Invoice_Quarter",
    "Invoice_Month",
    "Month_Number",
    "Month_Name",
    "Transaction_Hour",
    "City",
    "Store_Format",
    "Category",
    "Brand",
    "Channel",
    "Payment_Mode",
    "Units",
    "Cost_Price",
    "Selling_Price",
    "Revenue",
    "Cost",
    "Margin",
    "Margin_Pct",
    "Stock_On_Hand",
    "Reorder_Level",
    "Stock_Buffer",
    "Reorder_Flag",
    "Inventory_Status",
    "Lead_Time_Days",
    "Customer_Age",
    "Age_Group",
    "Customer_Gender",
    "Loyalty_Flag",
    "Loyalty_Status"
]

clean_df = clean_df[column_order]

print("Final column count:", len(clean_df.columns))
display(clean_df.head())

Final column count: 34


,Transaction_ID,Invoice_ID,Invoice_Date,Invoice_Date_Only,Invoice_Year,Invoice_Quarter,Invoice_Month,Month_Number,Month_Name,Transaction_Hour,...,Reorder_Level,Stock_Buffer,Reorder_Flag,Inventory_Status,Lead_Time_Days,Customer_Age,Age_Group,Customer_Gender,Loyalty_Flag,Loyalty_Status
0,TXN_000001,58018430,2024-02-02 13:50:00,2024-02-02,2024,2024Q1,2024-02,2,February,13,...,31,123,0,Sufficient Stock,11,20,18-24,Male,1,Member
1,TXN_000002,48157952,2024-10-09 11:52:00,2024-10-09,2024,2024Q4,2024-10,10,October,11,...,24,106,0,Sufficient Stock,5,26,25-34,Female,1,Member
2,TXN_000003,23283831,2024-08-26 22:03:00,2024-08-26,2024,2024Q3,2024-08,8,August,22,...,44,226,0,Sufficient Stock,8,27,25-34,Female,0,Non-Member
3,TXN_000004,53537460,2024-06-09 04:34:00,2024-06-09,2024,2024Q2,2024-06,6,June,4,...,63,340,0,Sufficient Stock,8,<NA>,Unknown,Female,0,Non-Member
4,TXN_000005,55348596,2024-06-07 01:13:00,2024-06-07,2024,2024Q2,2024-06,6,June,1,...,40,326,0,Sufficient Stock,9,60,55-64,Female,1,Member


## 9. Final Data-Quality Validation

The cleaned dataset is checked before export. The validation confirms that no rows were lost, the surrogate key is unique, expected calculated relationships remain correct, and only the intentionally retained age values remain null.

In [25]:
final_checks = pd.Series({
    "Rows retained": len(clean_df) == len(df),
    "Transaction ID unique": clean_df["Transaction_ID"].is_unique,
    "No missing transaction IDs": clean_df["Transaction_ID"].isna().sum() == 0,
    "No exact duplicate rows": clean_df.duplicated().sum() == 0,

    "Revenue formula valid": np.isclose(
        clean_df["Revenue"],
        clean_df["Units"] * clean_df["Selling_Price"],
        atol=0.01
    ).all(),

    "Cost formula valid": np.isclose(
        clean_df["Cost"],
        clean_df["Units"] * clean_df["Cost_Price"],
        atol=0.01
    ).all(),

    "Margin formula valid": np.isclose(
        clean_df["Margin"],
        clean_df["Revenue"] - clean_df["Cost"],
        atol=0.01
    ).all(),

    "Margin percentage valid": np.isclose(
        clean_df["Margin_Pct"],
        clean_df["Margin"] / clean_df["Revenue"],
        atol=0.0001
    ).all()
})

display(final_checks.to_frame("Passed"))

final_missing_summary = clean_df.isna().sum()
display(
    final_missing_summary[
        final_missing_summary > 0
    ].to_frame("Missing_Count")
)

,Passed
Rows retained,True
Transaction ID unique,True
No missing transaction IDs,True
No exact duplicate rows,True
Revenue formula valid,True
Cost formula valid,True
Margin formula valid,True
Margin percentage valid,True


,Missing_Count
Customer_Age,40081


## 10. Export the Cleaned Dataset

The cleaned dataset is exported as a new CSV file for SQL import. The raw source file remains unchanged.

In [28]:
from pathlib import Path

cleaned_folder = Path(
    r"C:\Users\janakiram\Documents\Data_Analytics_Portfolio\PROJECTS\Retail_Data_Integration\data\cleaned"
)

cleaned_folder.mkdir(
    parents=True,
    exist_ok=True
)

cleaned_file_path = (
    cleaned_folder / "fmcg_retail_cleaned_2024.csv"
)

export_df = clean_df.copy()

export_df["Invoice_Date"] = (
    export_df["Invoice_Date"]
    .dt.strftime("%Y-%m-%d %H:%M:%S")
)

export_df["Invoice_Date_Only"] = (
    export_df["Invoice_Date_Only"]
    .dt.strftime("%Y-%m-%d")
)

# Match the CSV headers with the MySQL column names.
export_df.columns = export_df.columns.str.lower()

# Use 0 temporarily for missing ages during the MySQL import.
export_df["customer_age"] = export_df["customer_age"].fillna(0)

export_df.to_csv(
    cleaned_file_path,
    index=False
)

print("Cleaned file exported to:")
print(cleaned_file_path)

Cleaned file exported to:
C:\Users\janakiram\Documents\Data_Analytics_Portfolio\PROJECTS\Retail_Data_Integration\data\cleaned\fmcg_retail_cleaned_2024.csv


## Cleaning Outcome

The Python cleaning process retained all 100,000 records and produced 34 analysis-ready columns.

Key transformations included:

- Creating a unique surrogate transaction key
- Converting invoice timestamps to datetime
- Standardizing categorical text and customer-gender labels
- Retaining missing ages without unsupported imputation
- Creating age groups and loyalty-status labels
- Adding date-analysis fields
- Creating inventory status, reorder flag, and stock buffer
- Validating all financial calculations
- Exporting a clean CSV for SQL integration

All final validation checks passed. The only remaining null values are the 40,081 intentionally retained missing customer ages.

## Loading Cleaned Data into MySQL

The cleaned retail dataset is loaded into MySQL using Python. A database connection provides a reliable alternative to the MySQL Workbench CSV Import Wizard and demonstrates integration between Python and SQL.

In [31]:
%pip install sqlalchemy mysql-connector-python

Note: you may need to restart the kernel to use updated packages.


### Establishing the MySQL Connection

A SQLAlchemy connection engine is created for the `retail_data_integration` database. The MySQL password is entered securely at runtime instead of being stored directly in the notebook.

In [33]:
from getpass import getpass
from urllib.parse import quote_plus
from sqlalchemy import create_engine, text

mysql_user = "root"
mysql_host = "localhost"
mysql_port = 3306
mysql_database = "retail_data_integration"

mysql_password = getpass("Enter your MySQL password: ")
encoded_password = quote_plus(mysql_password)

engine = create_engine(
    f"mysql+mysqlconnector://{mysql_user}:{encoded_password}"
    f"@{mysql_host}:{mysql_port}/{mysql_database}"
)

with engine.connect() as connection:
    result = connection.execute(
        text("SELECT DATABASE(), VERSION();")
    ).fetchone()

print("Connected database:", result[0])
print("MySQL version:", result[1])

Enter your MySQL password:  ········


Connected database: retail_data_integration
MySQL version: 8.0.44


In [34]:
# Prepare a database-ready copy.
db_df = clean_df.copy()
db_df.columns = db_df.columns.str.lower()

# Convert missing customer ages into genuine SQL NULL values.
db_df["customer_age"] = (
    db_df["customer_age"]
    .astype(object)
    .where(db_df["customer_age"].notna(), None)
)

# Prevent accidental duplicate loading.
with engine.connect() as connection:
    existing_rows = connection.execute(
        text("SELECT COUNT(*) FROM fmcg_retail_sales;")
    ).scalar()

print("Existing MySQL rows:", existing_rows)

if existing_rows == 0:
    db_df.to_sql(
        name="fmcg_retail_sales",
        con=engine,
        if_exists="append",
        index=False,
        chunksize=1000,
        method="multi"
    )

    print("Data loading completed.")
else:
    print("Import stopped because the table already contains data.")

Existing MySQL rows: 0


DataError: (mysql.connector.errors.DataError) 1416 (22003): Cannot get geometry object from data you send to the GEOMETRY field
[SQL: INSERT INTO fmcg_retail_sales (transaction_id, invoice_id, invoice_date, invoice_date_only, invoice_year, invoice_quarter, invoice_month, month_number, month_name, transaction_hour, city, store_format, category, brand, channel, payment_mode, units, cost_price, selling_price, revenue, cost, margin, margin_pct, stock_on_hand, reorder_level, stock_buffer, reorder_flag, inventory_status, lead_time_days, customer_age, age_group, customer_gender, loyalty_flag, loyalty_status) VALUES (%(transaction_id_m0)s, %(invoice_id_m0)s, %(invoice_date_m0)s, %(invoice_date_only_m0)s, %(invoice_year_m0)s, %(invoice_quarter_m0)s, %(invoice_month_m0)s, %(month_number_m0)s, %(month_name_m0)s, %(transaction_hour_m0)s, %(city_m0)s, %(store_format_m0)s, %(category_m0)s, %(brand_m0)s, %(channel_m0)s, %(payment_mode_m0)s, %(units_m0)s, %(cost_price_m0)s, %(selling_price_m0)s, %(rev